## MMR Re-ranking Engine

Implements Maximal Marginal Relevance (MMR) to re-rank candidate recommendations,
balancing relevance to the seed video against redundancy with already-selected videos.

Score(i) = λ · Relevance(i, query) - (1−λ) · max Similarity(i, already_selected)

At λ=1, this reduces to pure similarity ranking (baseline).
At λ=0, diversity is maximized regardless of relevance.

In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
df = pd.read_csv('/Users/anoushka/Desktop/Diversity-Aware-Recommender/data/cleaned_videos.csv')

In [5]:
embeddings_weighted = np.load('/Users/anoushka/Desktop/Diversity-Aware-Recommender/data/embeddings_weighted.npy')

In [6]:
embedding_sim_weighted = cosine_similarity(embeddings_weighted)

In [7]:
print(df.shape)
print(embedding_sim_weighted.shape)

(22427, 17)
(22427, 22427)


In [8]:
def mmr(seed_idx, similarity_matrix, lambda_param, top_n):
    """
    seed_idx: index of the seed video
    similarity_matrix: precomputed cosine similarity matrix (e.g. embedding_sim_weighted)
    lambda_param: the λ value (0 to 1)
    top_n: how many videos to recommend
    
    returns: list of selected video indices
    """
    selected = []
    candidates = [i for i in range(len(df)) if i!=seed_idx]
    
    while len(selected)<top_n:
        scores = {}
        for candidate in candidates:
            relevance = similarity_matrix[seed_idx, candidate]
            if len(selected) == 0:
                redundancy = 0
            else:
                redundancy = max([similarity_matrix[candidate, s] for s in selected])
            
            mmr_score = lambda_param * relevance - (1 - lambda_param) * redundancy
            scores[candidate] = mmr_score
        
        max_score_candidate = max(scores, key=scores.get)
        selected.append(max_score_candidate)
        candidates.remove(max_score_candidate)
    
    return selected
    





In [9]:
result = mmr(seed_idx=0, similarity_matrix=embedding_sim_weighted, lambda_param=0.5, top_n=10)

for idx in result:
    print(df.iloc[idx]['title'])

IS THIS THE CAMERA OF THE FUTURE?
Dan + Shay - Speechless (Wedding Video)
अगले 24 घंटे में आ सकता है आंधी तूफान- मौसम विभाग
World's First Engagement Ring Phone Case
Amazon.com commercial Xmas 1999Magnetic Whale Art
THE PROPOSAL | Felix & Marzia 💍
Samsung Galaxy S9 Impressions!
శ్రీదేవి కూతుర్ల కోసం ఆస్తుల విలువ|Sridevi Property For Her Daughters|Telugu Poster
Eminem speaks on working with Beyonce
Quality Audio Recording for Video


In [10]:
result = mmr(seed_idx=0, similarity_matrix=embedding_sim_weighted, lambda_param=0.9, top_n=10)

for idx in result:
    print(df.iloc[idx]['title'])

IS THIS THE CAMERA OF THE FUTURE?
THE PROPOSAL | Felix & Marzia 💍
looking back
PRODUCT PHOTOGRAPHY
Honest Husband
राजस्थानी बरात फौजी स्टूडियो झुन्झुनू | FOUJI STUDIO JHUNJHUNU | New Marwadi Video 2018
மணமேடையில் இந்த மணப்பெண்ணுக்கு நடந்த கொடுமையை பாருங்க! | Tamil News | Tamil Seithigal |
How I Make Videos
లైవ్ లో ముద్దిచ్చి పిచ్చెక్కిచింది  | Kirrak Party Team soup Game | Nikhil | NewsQube
To Our Daughter


### Initial Testing

Running MMR at λ=1.0, 0.9, and 0.5 on the same seed video to sanity-check behavior
before running a full systematic sweep.

- **λ=1.0** matches Day 5 baseline exactly (pure similarity), dominated by same-channel videos
- **λ=0.9** nearly identical to baseline, confirms relevance term dominates as expected at high λ
- **λ=0.5** visibly more diverse, includes genuinely unrelated content (weather, news, interviews)
  alongside on-topic wedding/proposal videos, demonstrating the relevance-diversity trade-off in action

In [12]:
lambdas_to_test = [0, 0.25, 0.5, 0.75, 1.0]

md_output = "## λ Sweep Results — Seed: \"WE WANT TO TALK ABOUT OUR MARRIAGE\"\n\n"

for lam in lambdas_to_test:
    result = mmr(seed_idx=0, similarity_matrix=embedding_sim_weighted, lambda_param=lam, top_n=10)
    md_output += f"### λ = {lam}\n"
    for idx in result:
        md_output += f"- {df.iloc[idx]['title']}\n"
    md_output += "\n"

print(md_output)

## λ Sweep Results — Seed: "WE WANT TO TALK ABOUT OUR MARRIAGE"

### λ = 0
- The Trump Presidency: Last Week Tonight with John Oliver (HBO)
- ആണുങ്ങള്‍ക്ക് എന്റെ ശരീരം മാത്രം മതി ..!! | Shweta Menon , Siddique , Biju Menon - Ithramathram
- Casually Laser-Exposing 0.2 mm PCB features on a 3D printer
- Why old buildings use the same leaf design
- Little Mix - Nothing Else Matters (Glory Days Tour)
- DIY Wireless Charger Install In Toyota Tacoma
- 2 Wounded In Shooting At LA Middle School, Suspect In Custody
- Heart-Wrenching Video: Starving Polar Bear on Iceless Land | National Geographic
- do you know how big my mousepad is?
- Revise effectively in last 4 days for prelims 2018 - Gain 10 -15 extra marks by Roman Saini

### λ = 0.25
- IS THIS THE CAMERA OF THE FUTURE?
- Granulated Sugar From Honey
- Mom of Two Ali Wong Has Suffered Enough
- Why old buildings use the same leaf design
- Jonny Brenns Sings Georgia by Vance Joy - Top 24 Solos - American Idol 2018 on ABC
- P!nk presentation on

In [13]:
with open('results_lambda_sweep.md', 'w', encoding='utf-8') as f:
    f.write(md_output)

print("Saved!")

Saved!


**Observation:** <br>As λ increases from 0 → 1, recommendations shift from
completely unrelated content (λ=0: Trump/John Oliver, PCB engraving, polar bear documentary)
toward increasingly wedding/marriage relevant content (λ=1: proposal videos, wedding ceremonies,
multilingual marriage content). λ=0.5 represents a genuine middle ground,  a mix of
on-topic (engagement ring, proposal) and off-topic diversity picks (weather report, Telugu news).

This confirms the MMR re-ranking behaves as theoretically expected across the full λ spectrum.

In [28]:
import random

random_idx = random.randint(0, len(df)-1)
print(random_idx, df.iloc[random_idx]['title'])

8465 CURRENT AFFAIRS | THE HINDU | 5th December 2017 | UPSC,IBPS, RRB, SSC,CDS,IB,CLAT


In [ ]:
seed_idx = random_idx  

lambdas_to_test = [0, 0.5, 1.0]

for lam in lambdas_to_test:
    print(f"\n=== λ = {lam} ===")
    result = mmr(seed_idx=seed_idx, similarity_matrix=embedding_sim_weighted, lambda_param=lam, top_n=10)
    for idx in result:
        print(df.iloc[idx]['title'])


=== λ = 0 ===
WE WANT TO TALK ABOUT OUR MARRIAGE
Portland-bound Amtrak train derails in Washington; at least 6 reported dead
Fresh Mint Mojito | फ्रेश मिन्ट मुहितो । Virgin Mojito Syrup
IELTS LISTENING PRACTICE TEST 2017 WITH ANSWERS | 27.11.2017
Levante vs Real Madrid 2-2 - All Goals & Extended Highlights - La Liga 03/02/2018 HD
If You Have Moles on One of These Places, It Has Surprising Meaning
Why old buildings use the same leaf design
The author of the 5:2 diet explains why eating healthy is more important than exercise
The Handmaid’s Tale Season 2 Teaser (Official) • The Handmaid's Tale on Hulu
When did you stop believing in Santa? (YIAY #386)

=== λ = 0.5 ===
CURRENT AFFAIRS | THE HINDU | 20th December 2017 | UPSC,IBPS, RRB, SSC,CDS,IB,CLAT
Pancakes Recipe | Quick and Easy Pancakes by Grandpa for Orphan Kids
ప్రామిసరీ నోట్ రాసేటప్పుడు అందరు చేసే తప్పు | What is Promissory Note | How To Write Promissory Note
Dr Anamika Amber | त्रिशूल लेकर राधे मां गोद में बैठके के मानीं, बेवफा ह

In [30]:
seeds_results = {
    'CURRENT AFFAIRS (echo chamber example)': {
        'seed_idx': 8465,
        'lambdas': [0, 0.5, 1.0]
    }
}

md_output = "## λ Sweep — Second Seed Validation\n\n"
md_output += "Seed: \"CURRENT AFFAIRS | THE HINDU | 5th December 2017\" — chosen to test a different content domain (educational/exam-prep) from the marriage seed.\n\n"

for seed_name, info in seeds_results.items():
    for lam in info['lambdas']:
        result = mmr(seed_idx=info['seed_idx'], similarity_matrix=embedding_sim_weighted, lambda_param=lam, top_n=10)
        md_output += f"### λ = {lam}\n"
        for idx in result:
            md_output += f"- {df.iloc[idx]['title']}\n"
        md_output += "\n"

md_output += "\n**Observation:** At λ=1.0, the recommender returns near-duplicate content from the same channel (different dates of the same daily current-affairs series) — a clear, concrete demonstration of the echo chamber problem this project addresses. At λ=0.5, the list breaks this loop, mixing relevant exam-prep content with genuinely unrelated topics (recipes, legal explainers, discussion shows). This confirms the λ=0.5 trade-off pattern observed on the marriage seed generalizes to a different content domain.\n"

print(md_output)

with open('results_lambda_sweep_seed2.md', 'w', encoding='utf-8') as f:
    f.write(md_output)

print("\nSaved!")

## λ Sweep — Second Seed Validation

Seed: "CURRENT AFFAIRS | THE HINDU | 5th December 2017" — chosen to test a different content domain (educational/exam-prep) from the marriage seed.

### λ = 0
- WE WANT TO TALK ABOUT OUR MARRIAGE
- Portland-bound Amtrak train derails in Washington; at least 6 reported dead
- Fresh Mint Mojito | फ्रेश मिन्ट मुहितो । Virgin Mojito Syrup
- IELTS LISTENING PRACTICE TEST 2017 WITH ANSWERS | 27.11.2017
- Levante vs Real Madrid 2-2 - All Goals & Extended Highlights - La Liga 03/02/2018 HD
- If You Have Moles on One of These Places, It Has Surprising Meaning
- Why old buildings use the same leaf design
- The author of the 5:2 diet explains why eating healthy is more important than exercise
- The Handmaid’s Tale Season 2 Teaser (Official) • The Handmaid's Tale on Hulu
- When did you stop believing in Santa? (YIAY #386)

### λ = 0.5
- CURRENT AFFAIRS | THE HINDU | 20th December 2017 | UPSC,IBPS, RRB, SSC,CDS,IB,CLAT
- Pancakes Recipe | Quick and Easy Pancakes

**Observation:** <br> At λ=1.0, the recommender returns near-duplicate content from the same channel (different dates of the same daily current-affairs series) , a clear, concrete demonstration of the echo chamber problem this project addresses. At λ=0.5, the list breaks this loop, mixing relevant exam-prep content with genuinely unrelated topics (recipes, legal explainers, discussion shows). This confirms the λ=0.5 trade-off pattern observed on the marriage seed generalizes to a different content domain.

In [31]:
%%time
result = mmr(seed_idx=0, similarity_matrix=embedding_sim_weighted, lambda_param=0.5, top_n=10)

CPU times: user 271 ms, sys: 356 ms, total: 627 ms
Wall time: 671 ms


**Performance note:** <br> A single MMR recommendation (top_n=10) over ~22,400 candidate
videos completes in ~670ms on a standard laptop CPU. While the algorithm's complexity
scales as O(top_n × num_candidates), this remains practical at the current dataset size.
At significantly larger scale (millions of videos), production systems would pre-filter
to a smaller candidate pool before applying MMR re-ranking.

Refactoring Code into class

In [8]:
class RecommendationEngine:
    def __init__(self, df, similarity_matrix):
        self.df = df
        self.similarity_matrix = similarity_matrix
    
    def recommend(self, seed_idx, lambda_param, top_n):
        selected = []
        candidates = [i for i in range(len(self.df)) if i != seed_idx]
        
        while len(selected) < top_n:
            scores = {}
            for candidate in candidates:
                relevance = self.similarity_matrix[seed_idx, candidate]
                if len(selected) == 0:
                    redundancy = 0
                else:
                    redundancy = max([self.similarity_matrix[candidate, s] for s in selected])
                
                mmr_score = lambda_param * relevance - (1 - lambda_param) * redundancy
                scores[candidate] = mmr_score
            
            max_score_candidate = max(scores, key=scores.get)
            selected.append(max_score_candidate)
            candidates.remove(max_score_candidate)
        
        return selected
    
    def get_titles(self, indices):
        return [self.df.iloc[idx]['title'] for idx in indices]

In [9]:
engine = RecommendationEngine(df, embedding_sim_weighted)

result = engine.recommend(seed_idx=0, lambda_param=0.5, top_n=10)
print(engine.get_titles(result))

['IS THIS THE CAMERA OF THE FUTURE?', 'Dan + Shay - Speechless (Wedding Video)', 'अगले 24 घंटे में आ सकता है आंधी तूफान- मौसम विभाग', "World's First Engagement Ring Phone Case", 'Amazon.com commercial Xmas 1999Magnetic Whale Art', 'THE PROPOSAL | Felix & Marzia 💍', 'Samsung Galaxy S9 Impressions!', 'శ్రీదేవి కూతుర్ల కోసం ఆస్తుల విలువ|Sridevi Property For Her Daughters|Telugu Poster', 'Eminem speaks on working with Beyonce', 'Quality Audio Recording for Video']
